In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../data/accidents.db")

def sql(requete):
    """Exécute une requête SQL et renvoie le résultat sous forme de DataFrame."""
    return pd.read_sql(requete, con)

In [2]:
sql("SELECT MIN(age), MAX(age), AVG(age) FROM usagers")

,MIN(age),MAX(age),AVG(age)
0,0.0,110.0,38.924589


In [3]:
# Pour chaque heure : regrouper les accidents de cette heure et les compter
sql("""
    SELECT heure, COUNT(*) AS nb_accidents
    FROM accidents
    GROUP BY heure
    ORDER BY heure
""")

,heure,nb_accidents
0,0,1168
1,1,955
2,2,807
3,3,602
4,4,630
5,5,886
6,6,1218
7,7,2442
8,8,3161
9,9,2673


In [161]:
# Pour chaque jour de la semaine
sql("""
    SELECT jour_semaine, COUNT(*) AS nb_accidents
    FROM accidents
    GROUP BY jour_semaine
    ORDER BY nb_accidents desc
""")

,jour_semaine,nb_accidents
0,Vendredi,8976
1,Jeudi,8112
2,Samedi,7789
3,Mardi,7757
4,Mercredi,7691
5,Lundi,7417
6,Dimanche,6660


In [162]:
# Pour chaque mois de l'année
sql("""
    SELECT mois, COUNT(*) AS nb_accidents
    FROM accidents
    GROUP BY mois
    ORDER BY mois
""")

,mois,nb_accidents
0,1,4069
1,2,3792
2,3,4370
3,4,4586
4,5,4517
5,6,5135
6,7,5209
7,8,4450
8,9,4620
9,10,4803


In [163]:
# Les 10 départements avec le plus d'accidents
sql("""
    SELECT dep, COUNT(*) AS nb_accidents
    FROM accidents
    GROUP BY dep
    ORDER BY nb_accidents DESC
    LIMIT 10
""")

,dep,nb_accidents
0,75,4191
1,93,2640
2,92,2485
3,13,2120
4,94,1963
5,69,1627
6,91,1348
7,33,1202
8,77,1068
9,59,1023


In [164]:
sql("SELECT agglomeration, COUNT(*) AS nb_accidents FROM accidents GROUP BY agglomeration")

,agglomeration,nb_accidents
0,En agglomération,34010
1,Hors agglomération,20392


In [165]:
sql("""
    SELECT meteo, COUNT(*) AS nb_accidents
    FROM accidents
    GROUP BY meteo
    ORDER BY nb_accidents DESC
""")

,meteo,nb_accidents
0,Normale,41802
1,Pluie légère,6779
2,Temps couvert,2374
3,Pluie forte,1449
4,Temps éblouissant,817
5,Brouillard / fumée,529
6,Autre,250
7,Neige / grêle,233
8,Vent fort / tempête,169


In [166]:
sql("""
    SELECT periode, COUNT(*) AS nb_accidents,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) as pct
    FROM accidents
    GROUP BY periode
    ORDER BY nb_accidents desc
""")

,periode,nb_accidents,pct
0,Jour,35580,65.40
1,Nuit,15223,27.98
2,Crépuscule,3599,6.62


In [167]:
sql("""
    SELECT
        CASE
            WHEN age < 18 THEN '0-17'
            WHEN age < 25 THEN '18-24'
            WHEN age < 45 THEN '25-44'
            WHEN age < 65 THEN '45-64'
            ELSE '65+'
        END AS tranche_age,
        COUNT(*) AS nb_usagers,
        SUM(CASE WHEN gravite = 'Tué' THEN 1 ELSE 0 END) AS nb_tues
    FROM usagers
    WHERE age IS NOT NULL
    GROUP BY tranche_age
    ORDER BY tranche_age
""")

,tranche_age,nb_usagers,nb_tues
0,0-17,11447,131
1,18-24,23366,567
2,25-44,43745,950
3,45-64,29729,837
4,65+,14321,947


In [168]:
sql("""
    SELECT
        a.periode,
        COUNT(*) AS nb_usagers,
        SUM(CASE WHEN u.gravite = 'Tué' THEN 1 ELSE 0 END) AS nb_tues,
        ROUND(100.0 * SUM(CASE WHEN u.gravite = 'Tué' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_tues
    FROM usagers AS u
    JOIN accidents AS a ON u.Num_Acc = a.Num_Acc
    GROUP BY a.periode
    ORDER BY pct_tues DESC
""")

,periode,nb_usagers,nb_tues,pct_tues
0,Nuit,35305,1255,3.55
1,Crépuscule,8163,244,2.99
2,Jour,81719,1933,2.37


In [169]:
sql("""
    SELECT
        a.agglomeration,
        COUNT(*) AS nb_usagers,
        SUM(CASE WHEN u.gravite = 'tué' THEN 1 ELSE 0 END) AS nb_tues,
        ROUND(100.0 * SUM(CASE WHEN u.gravite = 'Tué' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_tues
    FROM usagers AS u
    JOIN accidents AS a ON u.Num_acc = a.Num_acc
    GROUP BY a.agglomeration
    ORDER BY pct_tues DESC
""")

,agglomeration,nb_usagers,nb_tues,pct_tues
0,Hors agglomération,49522,0,4.69
1,En agglomération,75665,0,1.47


In [170]:
# Accidents par heure
import plotly.express as px

par_heure = sql("SELECT heure, COUNT(*) AS nb_accidents FROM accidents GROUP BY heure ORDER BY heure")

fig = px.bar(
    par_heure,                       # le tableau à représenter
    x="heure", y="nb_accidents",     # les colonnes pour chaque axe
    title="Nombre d'accidents selon l'heure (2024)",
    labels={"heure": "Heure de la journée", "nb_accidents": "Accidents"},  # noms affichés
)
fig.show()

In [171]:
# Accidents par jour de la semaine
par_jour = sql("SELECT jour_semaine, COUNT(*) AS nb_accidents FROM accidents GROUP BY jour_semaine")

jours = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
fig = px.bar(
    par_jour, x="jour_semaine", y="nb_accidents",
    category_orders={"jour_semaine": jours},   # sinon plotly trie les jours par ordre alphabétique
    title="Nombre d'accidents selon le jour de la semaine (2024)",
    labels={"jour_semaine": "", "nb_accidents": "Accidents"},
)
fig.show()

In [175]:
# Part de tués selon la période
gravite_periode = sql("""
    SELECT a.periode,
        ROUND(100.0 * SUM(CASE WHEN u.gravite = 'Tué' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_tues
    FROM usagers AS u
    JOIN accidents AS a ON u.Num_Acc = a.Num_Acc
    GROUP BY a.periode
""")

fig = px.bar(
    gravite_periode, x="periode", y="pct_tues",
    text="pct_tues",   # affiche la valeur au-dessus de chaque barre
    title="Part des usagers tués selon la période (2024)",
    labels={"periode": "", "pct_tues": "% d'usagers tués"},
)
fig.show()

In [176]:
par_age = sql("""
    SELECT
        CASE
            WHEN age < 18 THEN '0-17'
            WHEN age < 25 THEN '18-24'
            WHEN age < 45 THEN '25-44'
            WHEN age < 65 THEN '45-64'
            ELSE '65+'
        END AS tranche_age,
        COUNT(*) AS nb_usagers,
        SUM(CASE WHEN gravite = 'Tué' THEN 1 ELSE 0 END) AS nb_tues,
        ROUND(100.0 * SUM(CASE WHEN gravite = 'Tué' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_tues
    FROM usagers
    WHERE age IS NOT NULL
    GROUP BY tranche_age
    ORDER BY tranche_age
""")

fig = px.bar(
    par_age, x="tranche_age", y="pct_tues",
    text="pct_tues",   # affiche la valeur au-dessus de chaque barre
    title="Part d'usagers tués par tranche d'âge (2024)",
    labels={"tranche_age": "Âge", "pct_tues": "% d'usagers tués"},
)
fig.show()

In [177]:
par_agglomeration = sql("""
    SELECT
        a.agglomeration,
        COUNT(*) AS nb_usagers,
        SUM(CASE WHEN u.gravite = 'Tué' THEN 1 ELSE 0 END) AS nb_tues,
        ROUND(100.0 * SUM(CASE WHEN u.gravite = 'Tué' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_tues
    FROM usagers AS u
    JOIN accidents AS a ON u.Num_acc = a.Num_acc
    GROUP BY a.agglomeration
    ORDER BY pct_tues DESC
""")

fig = px.bar(
    par_agglomeration, x="agglomeration", y="pct_tues",
    text="pct_tues",   # affiche la valeur au-dessus de chaque barre
    title="Pourcentage d'usagers tués par type d'agglomération (2024)",
    labels={"agglomeration": "Agglomeration", "pct_tues": "% d'usagers tués"},
)
fig.show()